In [16]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback
import time
import random


In [ ]:
class CarRacingRewardWrapper(gym.Wrapper):
    def __init__(self, env, fase=1):
        super().__init__(env)
        self.fase_actual = fase
        self.reset_estadisticas()
        self.episode_count = 0
        
    def reset_estadisticas(self):
        self.pasos_en_verde = 0
        self.pasos_en_gris = 0
        self.ultima_posicion = None
        self.steps_sin_movimiento = 0
        self.total_steps = 0
        self.acciones_positivas = 0
        self.acciones_negativas = 0
        
    def analizar_colores(self, obs):
        try:
            region = obs[70:85, 38:58]
            
            verde_count = 0
            gris_count = 0
            total = max(region.shape[0] * region.shape[1], 1)
            
            for i in range(region.shape[0]):
                for j in range(region.shape[1]):
                    r, g, b = region[i, j]
                    r, g, b = int(r), int(g), int(b)
                    
                    diff_rg = abs(r - g)
                    diff_gb = abs(g - b)
                    if diff_rg < 25 and diff_gb < 25 and 90 < r < 120:
                        gris_count += 1
                    elif g > r + 20 and g > b + 20 and g > 95:
                        verde_count += 1
            
            return gris_count / total, verde_count / total
        except:
            return 0.0, 0.0
    
    def detectar_movimiento(self, obs):
        if self.ultima_posicion is None:
            self.ultima_posicion = obs.copy()
            return True
            
        try:
            diff = np.mean(np.abs(obs - self.ultima_posicion))
            self.ultima_posicion = obs.copy()
            return diff > 1.5
        except:
            return True
    
    def step(self, action):
        try:
            obs, reward, terminated, truncated, info = self.env.step(action)
            self.total_steps += 1
            
            gris_pct, verde_pct = self.analizar_colores(obs)
            se_mueve = self.detectar_movimiento(obs)
            
            custom_reward = 0.0
            
            #  SISTEMA DE RECOMPENSAS POR FASE
            if self.fase_actual == 1:
                # FASE 1: RECOMPENSAS SIMPLES Y GENEROSAS
                if se_mueve:
                    custom_reward += 1.0  # Generosa por movimiento
                    self.steps_sin_movimiento = 0
                else:
                    self.steps_sin_movimiento += 1
                    if self.steps_sin_movimiento > 10:
                        custom_reward -= 0.5
                
                if gris_pct > 0.3:  # Umbral bajo
                    custom_reward += 2.0  # Generosa por camino
                    self.pasos_en_gris += 1
                elif verde_pct > 0.25:
                    custom_reward -= 1.0  # Penalización suave
                    
                # Cualquier aceleración es buena en Fase 1
                if action[1] > 0.1:
                    custom_reward += 0.3
                    
            else:
                # FASE 2: RECOMPENSAS MÁS EXIGENTES
                if se_mueve:
                    custom_reward += 0.8  # Menos generosa
                    self.steps_sin_movimiento = 0
                else:
                    self.steps_sin_movimiento += 1
                    if self.steps_sin_movimiento > 8:  # Más exigente
                        custom_reward -= 0.8
                
                if gris_pct > 0.4:  # Umbral más alto
                    custom_reward += 1.5  # Menos generosa
                    self.pasos_en_gris += 1
                    
                    # Bonus por aceleración ÓPTIMA (no cualquier aceleración)
                    if 0.3 < action[1] < 0.7:
                        custom_reward += 0.8
                        
                elif verde_pct > 0.2:  # Más sensible al verde
                    custom_reward -= 1.2  # Penalización más fuerte
                
                # Recompensa por giros SUAVES (no cualquier giro)
                if 0.1 < abs(action[0]) < 0.5:
                    custom_reward += 0.4
            
            # LOGROS (comunes a ambas fases pero con diferentes umbrales)
            if self.fase_actual == 1:
                if self.pasos_en_gris >= 10:  # Fácil en Fase 1
                    custom_reward += 2.0
                    print(f" FASE 1: ¡Buen comienzo! {self.pasos_en_gris} pasos en gris")
                    self.pasos_en_gris = 0
            else:
                if self.pasos_en_gris >= 20:  # Más difícil en Fase 2
                    custom_reward += 3.0
                    print(f" FASE 2: ¡Conducción avanzada! {self.pasos_en_gris} pasos en gris")
                    self.pasos_en_gris = 0
            
            # LIMITAR recompensas
            custom_reward = np.clip(custom_reward, -3.0, 3.0).item()
            
            # Mostrar progreso
            if self.total_steps % 100 == 0:
                print(f" Fase {self.fase_actual} - Paso {self.total_steps}: Reward={custom_reward:.1f}")
            
            return obs, float(custom_reward), bool(terminated), bool(truncated), info
            
        except Exception as e:
            print(f"Error en step: {e}")
            return np.zeros((96, 96, 3)), -1.0, True, True, {}
    
    def reset(self, **kwargs):
        self.episode_count += 1
        print(f"\n FASE {self.fase_actual} - EPISODIO {self.episode_count}")
        self.reset_estadisticas()
        try:
            return self.env.reset(**kwargs)
        except:
            return np.zeros((96, 96, 3)), {}

class VisualizadorCallback(BaseCallback):
    def __init__(self, fase=1, verbose=0):
        super().__init__(verbose)
        self.fase = fase
        self.episodio = 0
        self.recompensa_episodio = 0
        self.mejor_recompensa = -float('inf')
        
    def _on_step(self):
        try:
            reward = self.locals['rewards'][0]
            if not np.isnan(reward):
                self.recompensa_episodio += reward
            
            if self.locals['dones'][0]:
                self.episodio += 1
                
                if self.recompensa_episodio > self.mejor_recompensa:
                    self.mejor_recompensa = self.recompensa_episodio
                    print(f" FASE {self.fase} - EPISODIO {self.episodio} - NUEVO RÉCORD: {self.recompensa_episodio:.1f}")
                else:
                    print(f" FASE {self.fase} - EPISODIO {self.episodio} - Recompensa: {self.recompensa_episodio:.1f}")
                
                self.recompensa_episodio = 0
            
            return True
        except Exception as e:
            print(f"Error")
            return True

def entrenar_fase_1():
    """FASE 1: Exploración y aprendizaje básico"""
    print(" INICIANDO FASE 1 - EXPLORACIÓN Y APRENDIZAJE BÁSICO")
    print(" Objetivo: Descubrir acciones básicas que funcionan")
    print(" Características: Recompensas generosas, alta exploración")
    
    env = gym.make("CarRacing-v3", render_mode="human", continuous=True)
    env = CarRacingRewardWrapper(env, fase=1)
    
    # CONFIGURACIÓN FASE 1
    model = PPO(
        "MlpPolicy",
        env,
        verbose=1,
        learning_rate=2.5e-4,    #  Rápido para aprender básicos
        n_steps=2048,            #  Mucha exploración
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,           #  ALTA exploración
        vf_coef=0.5,
        max_grad_norm=0.8,
        policy_kwargs=dict(
            net_arch=[64, 64]    # Red pequeña para patrones básicos
        )
    )
    
    callback = VisualizadorCallback(fase=1)
    print(" COMIENZA FASE 1...")
    model.learn(total_timesteps=10000, callback=callback)
    
    model.save("carracing_fase1")
    print(" FASE 1 COMPLETADA - Modelo guardado")
    
    env.close()
    return model

def entrenar_fase_2(modelo_fase1=None):
    """FASE 2: Refinamiento y optimización"""
    print("\n" + "="*60)
    print(" INICIANDO FASE 2 - REFINAMIENTO Y OPTIMIZACIÓN")
    print(" Objetivo: Mejorar y refinar las estrategias aprendidas")
    print(" Características: Recompensas exigentes, menos exploración")
    
    env = gym.make("CarRacing-v3", render_mode="human", continuous=True)
    env = CarRacingRewardWrapper(env, fase=2)
    
    # CONFIGURACIÓN FASE 2
    model = PPO(
        "MlpPolicy",
        env,
        verbose=1,
        learning_rate=1e-4,      # Más lento para ajustes finos
        n_steps=1024,            # Menos exploración
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.15,
        ent_coef=0.005,          # MENOS exploración
        vf_coef=0.5,
        max_grad_norm=0.6,
        policy_kwargs=dict(
            net_arch=[128, 128]  # Red más grande para patrones complejos
        )
    )
    
    # CARGAR PESOS DE FASE 1 SI EXISTE
    if modelo_fase1 is not None:
        print(" Cargando conocimientos de FASE 1...")
        model.set_parameters(modelo_fase1.get_parameters())
    
    callback = VisualizadorCallback(fase=2)
    print(" COMIENZA FASE 2...")
    model.learn(total_timesteps=15000, callback=callback)
    
    model.save("carracing_fase2")
    print(" FASE 2 COMPLETADA - Modelo guardado")
    
    env.close()
    return model

def probar_modelo(fase=2):
    """Probar el modelo entrenado"""
    print(f" PROBANDO MODELO FASE {fase}...")
    
    try:
        env = gym.make("CarRacing-v3", render_mode="human", continuous=True)
        env = CarRacingRewardWrapper(env, fase=fase)
        
        if fase == 1:
            model = PPO.load("carracing_fase1")
        else:
            model = PPO.load("carracing_fase2")
            
        print(f" Modelo FASE {fase} cargado")
        
        for episodio in range(3):
            obs, info = env.reset()
            done = False
            total_reward = 0
            pasos = 0
            
            print(f" EPISODIO PRUEBA {episodio + 1}")
            
            while not done and pasos < 500:
                action, _ = model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = env.step(action)
                total_reward += reward
                pasos += 1
                done = terminated or truncated
                
                time.sleep(0.02)
            
            print(f"    Pasos: {pasos} | Recompensa: {total_reward:.1f}")
        
        env.close()
        
    except Exception as e:
        print(f" Error: {e}")

#  PROGRAMA PRINCIPAL
if __name__ == "__main__":
    print("\n" + "="*70)
    print("🏎️  CARRACING - ENTRENAMIENTO POR FASES COMPLETO")
    print("="*70)
    
    print("\n OPCIONES DISPONIBLES:")
    print("1.  FASE 1 sola (10,000 pasos) - Exploración básica")
    print("2.  FASE 2 sola (15,000 pasos) - Refinamiento") 
    print("3.  FASE 1 + FASE 2 completo (25,000 pasos)")
    print("4.  Probar FASE 1")
    print("5.  Probar FASE 2")
    
    opcion = input("\nElige opción (1-5): ").strip()
    
    if opcion == "1":
        entrenar_fase_1()
        
    elif opcion == "2":
        try:
            modelo_fase1 = PPO.load("carracing_fase1")
            entrenar_fase_2(modelo_fase1)
        except:
            print("❌ Necesitas entrenar FASE 1 primero o usar opción 3")
            
    elif opcion == "3":
        print(" INICIANDO ENTRENAMIENTO COMPLETO POR FASES")
        print("="*50)
        modelo_fase1 = entrenar_fase_1()
        print("\n" + "="*50)
        print("TRANSICIÓN: FASE 1 → FASE 2")
        print("="*50)
        modelo_fase2 = entrenar_fase_2(modelo_fase1)
        print("\n ENTRENAMIENTO COMPLETO POR FASES TERMINADO")
        
    elif opcion == "4":
        probar_modelo(fase=1)
        
    elif opcion == "5":
        probar_modelo(fase=2)
        
    else:
        print("Ejecutando entrenamiento completo por defecto...")
        modelo_fase1 = entrenar_fase_1()
        modelo_fase2 = entrenar_fase_2(modelo_fase1)
    
    print("\n PROGRAMA TERMINADO")


🏎️  CARRACING - ENTRENAMIENTO POR FASES COMPLETO

📊 OPCIONES DISPONIBLES:
1.  FASE 1 sola (10,000 pasos) - Exploración básica
2.  FASE 2 sola (15,000 pasos) - Refinamiento
3.  FASE 1 + FASE 2 completo (25,000 pasos)
4.  Probar FASE 1
5.  Probar FASE 2
 INICIANDO ENTRENAMIENTO COMPLETO POR FASES
 INICIANDO FASE 1 - EXPLORACIÓN Y APRENDIZAJE BÁSICO
 Objetivo: Descubrir acciones básicas que funcionan
 Características: Recompensas generosas, alta exploración
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.
 COMIENZA FASE 1...

 FASE 1 - EPISODIO 1
 FASE 1: ¡Buen comienzo! 10 pasos en gris
 FASE 1: ¡Buen comienzo! 10 pasos en gris
 FASE 1: ¡Buen comienzo! 10 pasos en gris
 FASE 1: ¡Buen comienzo! 10 pasos en gris
 FASE 1: ¡Buen comienzo! 10 pasos en gris
 FASE 1: ¡Buen comienzo! 10 pasos en gris
 FASE 1: ¡Buen comienzo! 10 pasos en gris
 FASE 1: ¡Buen comienzo! 10 pasos en gris
 Fase 1 - Paso 100: Reward=

RuntimeError: Error(s) in loading state_dict for ActorCriticPolicy:
	size mismatch for mlp_extractor.policy_net.0.weight: copying a param with shape torch.Size([64, 27648]) from checkpoint, the shape in current model is torch.Size([128, 27648]).
	size mismatch for mlp_extractor.policy_net.0.bias: copying a param with shape torch.Size([64]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for mlp_extractor.policy_net.2.weight: copying a param with shape torch.Size([64, 64]) from checkpoint, the shape in current model is torch.Size([128, 128]).
	size mismatch for mlp_extractor.policy_net.2.bias: copying a param with shape torch.Size([64]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for mlp_extractor.value_net.0.weight: copying a param with shape torch.Size([64, 27648]) from checkpoint, the shape in current model is torch.Size([128, 27648]).
	size mismatch for mlp_extractor.value_net.0.bias: copying a param with shape to